# Branch Operations: Tool Usage & Function Calling

This notebook explores lionagi's powerful function calling capabilities, showing how AI agents can interact with external tools and perform complex operations beyond text generation.

**Key Concepts:**
- **Tool Integration**: Connect AI agents to external functions and APIs
- **Automatic Function Selection**: Let the AI choose which tools to use based on context
- **Parallel Function Execution**: Handle multiple function calls simultaneously
- **Structured Action Flow**: Track function calls, parameters, and results

**Function Calling in Practice:**

The `operate()` method enables AI agents to analyze problems and automatically select appropriate tools to solve them. This bridges the gap between natural language understanding and concrete actions.

**How Function Calling Works:**
1. **Problem Analysis**: The AI examines the given context and instruction
2. **Tool Selection**: Based on available tools, the AI decides which functions to call
3. **Parameter Extraction**: The AI determines the correct parameters for each function
4. **Execution**: Functions are called with the AI-generated parameters
5. **Result Integration**: Function outputs are incorporated back into the conversation

This creates a powerful paradigm where AI agents can reason about problems and take concrete actions to solve them.

In [1]:
question = "There are [basketball, football, backpack, water bottle, strawberry, tennis ball, rockets]. each comes in four different colors, what is the number of unique kinds of ball?"

question2 = "There are three fruits in total, each with 2 different colors, how many unique kinds of fruits are there?"

context1 = {"Question1": question, "question2": question2}

In [2]:
# define a function


def multiply(number1: float, number2: float):
    """
    Perform multiplication on two numbers.

    Args:
        number1: First number to multiply.
        number2: Second number to multiply.

    Returns:
        The product of number1 and number2.

    """
    return number1 * number2

In [3]:
from lionagi import Branch, iModel

branch = Branch(
    system="you are asked to perform as a function picker and parameter provider",
    tools=multiply,
    chat_model=iModel(model="openai/gpt-4.1-mini"),
)

task = "Think step by step, understand the following basic math question and provide parameters for function calling. with parallel processing"

In [7]:
# request actions by flagging actions=True
out = await branch.operate(instruction=task, context=context1, actions=True, skip_validation=True)
print(branch.messages[1].rendered)

# print(out.action_requests)
# print(out.action_responses)

Instruction: Think step by step, understand the following basic math question and provide parameters for function calling. with parallel processing
Context:
- Question1: There are [basketball, football, backpack, water bottle, strawberry, tennis ball, rockets]. each comes in four different colors, what is the number of unique kinds of ball?
  question2: There are three fruits in total, each with 2 different colors, how many unique kinds of fruits are there?
ResponseFormat: "**ResponseSchema:**\n```json\n{\n  \"$defs\": {\n    \"ActionRequestModel\": {\n      \"additionalProperties\": false,\n      \"description\": \"Captures a single action request, typically from a user or system message.\\nIncludes the name of the function and the arguments to be passed.\",\n      \"properties\": {\n        \"function\": {\n          \"anyOf\": [\n            {\n              \"type\": \"string\"\n            },\n            {\n              \"type\": \"null\"\n            }\n          ],\n          

**Understanding the Output:**

The output shows two key components:

**Action Requests**: The AI's plan for solving the problems
- Question 1: Identified 8 types of balls × 4 colors = 32 total (but only balls, so multiply 2 × 4)
- Question 2: Identified 3 fruits × 2 colors = 6 total

**Action Responses**: The actual function execution results
- First calculation: 8 × 4 = 32 (total items that are balls)
- Second calculation: 3 × 2 = 6 (unique fruit combinations)

Notice how the AI automatically determined which calculations were needed and called the multiply function in parallel to solve both questions efficiently.

In [5]:
df = branch.to_df()

In [6]:
branch.messages[1].rendered

'Instruction: Think step by step, understand the following basic math question and provide parameters for function calling. with parallel processing\nContext:\n- Question1: There are [basketball, football, backpack, water bottle, strawberry, tennis ball, rockets]. each comes in four different colors, what is the number of unique kinds of ball?\n  question2: There are three fruits in total, each with 2 different colors, how many unique kinds of fruits are there?\nTools:\n- tools:\n  - type: function\n    function:\n      name: multiply\n      description: Perform multiplication on two numbers.\n      parameters:\n        type: object\n        properties:\n          number1:\n            type: number\n            description: First number to multiply.\n          number2:\n            type: number\n            description: Second number to multiply.\n        required:\n        - number1\n        - number2\nResponseFormat: "**ResponseSchema:**\\n```json\\n{\\n  \\"$defs\\": {\\n    \\"Acti